# Feature Correlation & Redundancy Analysis

**Purpose:** Identify redundant features that measure the same underlying signal.

**Problem:**
- Features may be predictive individually but redundant when combined
- Example: RSI and Distance_MA might both measure overextension
- Redundancy leads to multicollinearity in portfolio construction

**Solution:**
- Compute correlation matrix between features (not features vs returns)
- If |corr(A, B)| > 0.7 → drop the one with lower |IC|

**Date:** March 1, 2026  
**Asset:** EURUSD (Daily)

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.features.generators import (
    ma_spread,
    distance_from_ma,
    atr,
    rsi,
    rate_of_change,
    breakout_indicator,
    return_volatility_ratio,
    close_position_in_range,
    z_score_returns
)

from src.features.correlation_analysis import (
    compute_feature_correlation_matrix,
    identify_redundant_features,
    plot_feature_correlation,
    analyze_feature_clusters,
    print_redundancy_report,
    create_feature_summary_table
)

# Plotting settings
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    plt.style.use('default')
sns.set_palette('husl')

print("✓ Imports successful")

## 1. Load Data

In [ ]:
# Load EURUSD daily data
df = pd.read_csv('../data/raw/EURUSD_daily.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.set_index('timestamp').sort_index()
df.columns = df.columns.str.lower()

print(f"Data loaded: {len(df)} bars")
print(f"Date range: {df.index[0]} to {df.index[-1]}")
print(f"Columns: {df.columns.tolist()}")

## 2. Generate All Features

Generate the same features tested in notebook 05.

In [ ]:
prices = df['close']
high = df['high']
low = df['low']

# Generate all features
features = {}

# Moving average features
features['MA_Spread_50_200'] = ma_spread(prices, 50, 200)
features['Distance_MA_20'] = distance_from_ma(prices, 20)
features['Distance_MA_50'] = distance_from_ma(prices, 50)

# Momentum features
features['ROC_5'] = rate_of_change(prices, 5)
features['ROC_10'] = rate_of_change(prices, 10)
features['ROC_20'] = rate_of_change(prices, 20)

# Oscillators
features['RSI_14'] = rsi(prices, 14)
features['RSI_28'] = rsi(prices, 28)

# Volatility
features['ATR_14'] = atr(high, low, prices, 14)
features['Return_Vol_Ratio'] = return_volatility_ratio(prices, 20)

# Pattern/range features
features['Close_Position'] = close_position_in_range(high, low, prices)
features['Breakout_20'] = breakout_indicator(prices, 20)

# Statistical
features['ZScore_Returns'] = z_score_returns(prices, 20)

print(f"\n✓ Generated {len(features)} features")
print(f"\nFeature list:")
for i, name in enumerate(features.keys(), 1):
    print(f"  {i:2d}. {name}")

## 3. Compute Feature Correlation Matrix

In [ ]:
# Compute Spearman correlation between features
feature_corr = compute_feature_correlation_matrix(features)

print("Feature Correlation Matrix:")
print(feature_corr.round(2))

## 4. Visualize Feature Correlations

In [ ]:
# Create heatmap
plot_feature_correlation(
    feature_corr,
    save_path='../reports/figures/feature_correlation_matrix.png',
    figsize=(14, 12)
)

## 5. Identify Strongly Correlated Pairs

Find all pairs with |correlation| > 0.7

In [ ]:
# Find high correlation pairs
high_corr_pairs = []

for i in range(len(feature_corr)):
    for j in range(i + 1, len(feature_corr)):
        corr_val = feature_corr.iloc[i, j]
        if abs(corr_val) > 0.7:
            high_corr_pairs.append({
                'Feature_1': feature_corr.index[i],
                'Feature_2': feature_corr.columns[j],
                'Correlation': corr_val
            })

if high_corr_pairs:
    high_corr_df = pd.DataFrame(high_corr_pairs)
    high_corr_df = high_corr_df.sort_values('Correlation', key=abs, ascending=False)
    
    print("="*70)
    print(f"HIGHLY CORRELATED PAIRS (|corr| > 0.7)")
    print("="*70)
    print(high_corr_df.to_string(index=False))
else:
    print("\n✓ No highly correlated pairs found (all |corr| < 0.7)")

## 6. Load IC Scores from Previous Analysis

We need IC scores to decide which feature to keep when redundancy is detected.

In [ ]:
# IC scores from notebook 05 univariate tests
# These are the Information Coefficients (Spearman correlation with next-day returns)
ic_scores = {
    'Close_Position': -0.7530,       # Winner: extremely strong
    'Distance_MA_20': -0.0650,       # Winner: robust
    'Distance_MA_50': -0.0520,       # Similar to MA_20
    'MA_Spread_50_200': -0.0570,     # Winner: stable
    'ROC_5': -0.0430,                # Short-term momentum
    'ROC_10': -0.0540,               # Medium-term momentum
    'ROC_20': -0.0380,               # Longer-term momentum
    'RSI_14': -0.0540,               # Standard RSI
    'RSI_28': -0.0450,               # Longer RSI
    'ATR_14': -0.0120,               # Volatility only
    'Return_Vol_Ratio': -0.0210,     # Risk-adjusted return
    'Breakout_20': -0.0320,          # Breakout indicator
    'ZScore_Returns': -0.0080        # Standardized returns
}

# Sort by absolute IC
ic_sorted = sorted(ic_scores.items(), key=lambda x: abs(x[1]), reverse=True)

print("IC Scores (from Notebook 05):")
print("="*50)
for feat, ic in ic_sorted:
    print(f"{feat:25s}: {ic:>7.4f}")

## 7. Identify Redundant Features

Apply redundancy detection:
- Threshold: |correlation| > 0.7
- Rule: Drop the feature with lower |IC|

In [ ]:
# Identify redundant features
redundancy_info = identify_redundant_features(
    feature_corr,
    ic_scores,
    threshold=0.7
)

# Print detailed report
print_redundancy_report(redundancy_info, verbose=True)

## 8. Feature Summary Table

In [ ]:
# Create comprehensive summary table
summary_table = create_feature_summary_table(
    ic_scores,
    feature_corr,
    redundancy_info
)

print("\n" + "="*90)
print("FEATURE SUMMARY TABLE")
print("="*90)
print(summary_table.to_string(index=False))
print("\n" + "="*90)

## 9. Analyze Feature Clusters

Group features by similarity using hierarchical clustering.

In [ ]:
# Identify feature clusters (correlation > 0.5)
clusters = analyze_feature_clusters(feature_corr, threshold=0.5)

print("="*70)
print("FEATURE CLUSTERS (correlation > 0.5)")
print("="*70)

for cluster_id, cluster_features in sorted(clusters.items()):
    if len(cluster_features) > 1:  # Only show multi-feature clusters
        print(f"\n{cluster_id}:")
        for feat in cluster_features:
            ic = ic_scores.get(feat, 0)
            status = '✗ DROP' if feat in redundancy_info['to_drop'] else '✓ KEEP'
            print(f"  {status:8s} {feat:25s} (IC={ic:>7.4f})")

## 10. Final Recommendations

In [ ]:
# Features to keep
features_to_keep = [f for f in ic_scores.keys() if f not in redundancy_info['to_drop']]
features_to_drop = redundancy_info['to_drop']

print("="*80)
print("FINAL FEATURE SELECTION RECOMMENDATIONS")
print("="*80)

print(f"\n✓ KEEP ({len(features_to_keep)} features):")
for feat in sorted(features_to_keep, key=lambda x: abs(ic_scores[x]), reverse=True):
    ic = ic_scores[feat]
    max_corr = feature_corr.loc[feat].drop(feat).abs().max()
    print(f"  {feat:25s} IC={ic:>7.4f}  Max_Corr={max_corr:.3f}")

if features_to_drop:
    print(f"\n✗ DROP ({len(features_to_drop)} features):")
    for feat in features_to_drop:
        ic = ic_scores[feat]
        print(f"  {feat:25s} IC={ic:>7.4f}  (redundant)")
else:
    print("\n✓ No features need to be dropped - all are independent")

print("\n" + "="*80)
print(f"Feature reduction: {len(ic_scores)} → {len(features_to_keep)} features")
print(f"Reduction: {len(features_to_drop)} features ({100*len(features_to_drop)/len(ic_scores):.1f}%)")
print("="*80)

## 11. Correlation Analysis with Different Thresholds

In [ ]:
# Test different correlation thresholds
thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]

print("="*70)
print("SENSITIVITY ANALYSIS: Different Correlation Thresholds")
print("="*70)
print(f"{'Threshold':<12s} {'Features Dropped':<18s} {'Features Kept':<15s} {'Reduction %'}")
print("-"*70)

for thresh in thresholds:
    redund = identify_redundant_features(feature_corr, ic_scores, threshold=thresh)
    n_dropped = len(redund['to_drop'])
    n_kept = len(ic_scores) - n_dropped
    pct = 100 * n_dropped / len(ic_scores)
    print(f"{thresh:<12.1f} {n_dropped:<18d} {n_kept:<15d} {pct:>6.1f}%")

print("\nRecommendation: Use threshold = 0.7 (standard cutoff for multicollinearity)")

## 12. Export Results

In [ ]:
# Save correlation matrix
feature_corr.to_csv('../reports/feature_correlation_matrix.csv')
print("✓ Saved: reports/feature_correlation_matrix.csv")

# Save summary table
summary_table.to_csv('../reports/feature_redundancy_summary.csv', index=False)
print("✓ Saved: reports/feature_redundancy_summary.csv")

# Save final feature list
final_features = pd.DataFrame({
    'Feature': features_to_keep,
    'IC': [ic_scores[f] for f in features_to_keep],
    'Status': ['KEEP'] * len(features_to_keep)
})
final_features = final_features.sort_values('IC', key=abs, ascending=False)
final_features.to_csv('../reports/final_feature_list.csv', index=False)
print("✓ Saved: reports/final_feature_list.csv")

print("\n✓ All results exported successfully")